<a href="https://colab.research.google.com/github/MrT4ttoo/GestionInformacion/blob/main/Lab04_Ingesta_Datos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 04 — Ingesta de Datos desde Múltiples Fuentes
## Gestión de la Información — Universidad Tecnológica de Panamá
**Módulo III: Ingeniería de Datos | Semana 5**

In [2]:
import requests
import pandas as pd
import numpy as np
import json
import sqlite3
import os

Se importan todas las librerías necesarias: `requests` para APIs REST, `pandas` y `numpy` para manipulación de datos, `json` y `sqlite3` para fuentes locales, y `os` para operaciones del sistema de archivos.

## Paso 1: Extraer datos de una API REST

In [3]:
url = 'https://jsonplaceholder.typicode.com/users'

try:
    response = requests.get(url, timeout=10)
    if response.status_code == 200:
        data = response.json()
        df_api = pd.json_normalize(data)
        print(f"Datos de API: {df_api.shape}")
        print(df_api[['id', 'name', 'email', 'address.city']].head())
    else:
        print(f"Error HTTP: {response.status_code}")
        df_api = pd.DataFrame()
except requests.exceptions.RequestException as e:
    print(f"API no disponible: {e}. Usando datos de respaldo.")
    df_api = pd.DataFrame([
        {"id": 1, "name": "Leanne Graham",  "email": "sincere@april.biz",   "address.city": "Gwenborough"},
        {"id": 2, "name": "Ervin Howell",   "email": "shanna@melissa.tv",   "address.city": "Wisokyburgh"},
        {"id": 3, "name": "Clementine Bauch","email": "nathan@yesenia.net", "address.city": "McKenziehaven"},
        {"id": 4, "name": "Patricia Lebsack","email": "julianne@kory.org",  "address.city": "South Elvis"},
        {"id": 5, "name": "Chelsey Dietrich","email": "lucy@anna.tv",       "address.city": "Roscoeview"},
    ])
    print(f"Datos de API (respaldo): {df_api.shape}")
    print(df_api[['id', 'name', 'email', 'address.city']])

Datos de API: (10, 15)
   id              name                      email   address.city
0   1     Leanne Graham          Sincere@april.biz    Gwenborough
1   2      Ervin Howell          Shanna@melissa.tv    Wisokyburgh
2   3  Clementine Bauch         Nathan@yesenia.net  McKenziehaven
3   4  Patricia Lebsack  Julianne.OConner@kory.org    South Elvis
4   5  Chelsey Dietrich   Lucio_Hettinger@annie.ca     Roscoeview


La API de JSONPlaceholder retorna 10 usuarios con datos de contacto y dirección. Se usa `pd.json_normalize()` para aplanar los campos anidados (como `address.city`) y convertirlos en columnas planas del DataFrame.

In [4]:
url2 = 'https://restcountries.com/v3.1/region/americas?fields=name,capital,population,area'

try:
    resp2 = requests.get(url2, timeout=10)
    if resp2.status_code == 200:
        paises = resp2.json()
        df_paises = pd.json_normalize(paises)
        print(f"Datos de países: {df_paises.shape}")
        print(df_paises.head())
    else:
        print(f"Error HTTP: {resp2.status_code}")
        df_paises = pd.DataFrame()
except requests.exceptions.RequestException as e:
    print(f"API no disponible: {e}. Usando datos de respaldo.")
    df_paises = pd.DataFrame([
        {"name.common": "Panama",    "capital": ["Panama City"], "population": 4351000,   "area": 75417.0},
        {"name.common": "Colombia",  "capital": ["Bogotá"],      "population": 51049498,  "area": 1141748.0},
        {"name.common": "Mexico",    "capital": ["Mexico City"], "population": 128932753, "area": 1964375.0},
        {"name.common": "Brazil",    "capital": ["Brasília"],    "population": 212559417, "area": 8515767.0},
        {"name.common": "Argentina", "capital": ["Buenos Aires"],"population": 45376763,  "area": 2780400.0},
        {"name.common": "Costa Rica","capital": ["San José"],    "population": 5094118,   "area": 51100.0},
    ])
    print(f"Datos de países (respaldo): {df_paises.shape}")
    print(df_paises)

Datos de países: (56, 31)
           capital      area  population  \
0    [George Town]     264.0       84738   
1       [Plymouth]     102.0        4386   
2   [St. George's]     344.0      109021   
3        [Managua]  130373.0     6803886   
4  [Washington DC]      34.2           0   

                            name.common                         name.official  \
0                        Cayman Islands                        Cayman Islands   
1                            Montserrat                            Montserrat   
2                               Grenada                               Grenada   
3                             Nicaragua                 Republic of Nicaragua   
4  United States Minor Outlying Islands  United States Minor Outlying Islands   

           name.nativeName.eng.official            name.nativeName.eng.common  \
0                        Cayman Islands                        Cayman Islands   
1                            Montserrat                     

La API de REST Countries devuelve información geográfica de los países de las Américas. El campo `name` llega como objeto anidado, por lo que `json_normalize` lo expande como `name.common` y `name.official`, y `capital` aparece como lista.

## Paso 2: Leer datos de archivos CSV y JSON

In [5]:
np.random.seed(42)
n = 100

ventas = pd.DataFrame({
    'fecha': pd.date_range('2025-01-01', periods=n, freq='D'),
    'producto': np.random.choice(['Laptop', 'Tablet', 'Monitor', 'Teclado'], n),
    'cantidad': np.random.randint(1, 20, n),
    'precio_unitario': np.random.uniform(50, 2000, n).round(2),
    'region': np.random.choice(['Panama', 'Chiriqui', 'Cocle', 'Veraguas'], n),
})
ventas['total'] = ventas['cantidad'] * ventas['precio_unitario']
ventas.to_csv('ventas_simuladas.csv', index=False)

df_csv = pd.read_csv('ventas_simuladas.csv')
print(f"Datos CSV: {df_csv.shape}")
print(df_csv.head())

Datos CSV: (100, 6)
        fecha producto  cantidad  precio_unitario    region     total
0  2025-01-01  Monitor        18           837.48  Veraguas  15074.64
1  2025-01-02  Teclado        12           176.54    Panama   2118.48
2  2025-01-03   Laptop         2           545.14  Veraguas   1090.28
3  2025-01-04  Monitor        10           531.41    Panama   5314.10
4  2025-01-05  Monitor         4          1407.79  Veraguas   5631.16


Se generan 100 registros de ventas simuladas con `numpy` y se guardan en `ventas_simuladas.csv`. Al releer el archivo con `pd.read_csv()`, se confirma que el dataset tiene 100 filas y 6 columnas: fecha, producto, cantidad, precio unitario, región y total.

In [6]:
tiendas = [
    {"id": 1, "nombre": "Tienda Albrook",  "region": "Panama",   "empleados": 15},
    {"id": 2, "nombre": "Tienda David",    "region": "Chiriqui", "empleados": 8},
    {"id": 3, "nombre": "Tienda Penonome", "region": "Cocle",    "empleados": 6},
    {"id": 4, "nombre": "Tienda Santiago", "region": "Veraguas", "empleados": 7},
]

with open('tiendas.json', 'w') as f:
    json.dump(tiendas, f)

df_json = pd.read_json('tiendas.json')
print(f"Datos JSON: {df_json.shape}")
print(df_json)

Datos JSON: (4, 4)
   id           nombre    region  empleados
0   1   Tienda Albrook    Panama         15
1   2     Tienda David  Chiriqui          8
2   3  Tienda Penonome     Cocle          6
3   4  Tienda Santiago  Veraguas          7


El archivo `tiendas.json` contiene la configuración de 4 tiendas, una por cada región del dataset de ventas. Pandas lee el JSON directamente con `pd.read_json()` y lo convierte en un DataFrame de 4 filas y 4 columnas, listo para hacer el merge posterior.

## Paso 3: Conectar a una base de datos SQLite

In [7]:
conn = sqlite3.connect('ejemplo_gi.db')

df_clientes = pd.DataFrame({
    'id': range(1, 21),
    'nombre': [f'Cliente_{i}' for i in range(1, 21)],
    'region': np.random.choice(['Panama', 'Chiriqui', 'Cocle', 'Veraguas'], 20),
    'tipo': np.random.choice(['Premium', 'Regular', 'Nuevo'], 20),
})
df_clientes.to_sql('clientes', conn, if_exists='replace', index=False)
print("Tabla 'clientes' creada en SQLite.")

Tabla 'clientes' creada en SQLite.


Se crea la base de datos `ejemplo_gi.db` y se inserta una tabla `clientes` con 20 registros usando `to_sql()`. El parámetro `if_exists='replace'` garantiza que la tabla se recrea limpia en cada ejecución.

In [8]:
df_sql = pd.read_sql('SELECT * FROM clientes', conn)
print(f"Datos SQLite: {df_sql.shape}")
print(df_sql.head())
conn.close()

Datos SQLite: (20, 4)
   id     nombre    region     tipo
0   1  Cliente_1  Chiriqui  Premium
1   2  Cliente_2    Panama    Nuevo
2   3  Cliente_3  Chiriqui  Regular
3   4  Cliente_4  Chiriqui  Premium
4   5  Cliente_5  Chiriqui  Regular


Se consulta la tabla con SQL estándar a través de `pd.read_sql()`, que retorna el resultado directamente como DataFrame. Se cierra la conexión al terminar para liberar el recurso.

## Paso 4: Unificar datos y enriquecer

In [9]:
df_enriquecido = df_csv.merge(df_json, on='region', how='left')
print(f"Datos enriquecidos: {df_enriquecido.shape}")
print(df_enriquecido.head())

Datos enriquecidos: (100, 9)
        fecha producto  cantidad  precio_unitario    region     total  id  \
0  2025-01-01  Monitor        18           837.48  Veraguas  15074.64   4   
1  2025-01-02  Teclado        12           176.54    Panama   2118.48   1   
2  2025-01-03   Laptop         2           545.14  Veraguas   1090.28   4   
3  2025-01-04  Monitor        10           531.41    Panama   5314.10   1   
4  2025-01-05  Monitor         4          1407.79  Veraguas   5631.16   4   

            nombre  empleados  
0  Tienda Santiago          7  
1   Tienda Albrook         15  
2  Tienda Santiago          7  
3   Tienda Albrook         15  
4  Tienda Santiago          7  


Se realiza un `LEFT JOIN` entre las ventas y las tiendas usando `region` como clave común. El resultado agrega los campos `id`, `nombre` y `empleados` de cada tienda a cada transacción de venta, manteniendo todas las 100 filas originales.

In [10]:
resumen_region = df_csv.groupby('region').agg({
    'total': ['sum', 'mean', 'count'],
    'cantidad': 'sum'
}).round(2)

resumen_region.columns = ['venta_total', 'venta_promedio', 'num_transacciones', 'unidades_vendidas']
resumen_region = resumen_region.reset_index()
print("Resumen por región:")
print(resumen_region)

Resumen por región:
     region  venta_total  venta_promedio  num_transacciones  unidades_vendidas
0  Chiriqui    279548.56        10751.87                 26                228
1     Cocle    203126.85         8463.62                 24                198
2    Panama    359367.77        14973.66                 24                280
3  Veraguas    229025.76         8808.68                 26                216


Se agrupan las ventas por región para obtener métricas clave: ventas totales, promedio por transacción, número de transacciones y unidades vendidas. Este resumen permite comparar el rendimiento de cada tienda de forma rápida.

## Paso 5: Guardar en formato Parquet

In [11]:
df_enriquecido.to_parquet('datos_unificados.parquet', index=False)

df_verificacion = pd.read_parquet('datos_unificados.parquet')
print(f"Parquet leído correctamente: {df_verificacion.shape}")
print(df_verificacion.head())

Parquet leído correctamente: (100, 9)
        fecha producto  cantidad  precio_unitario    region     total  id  \
0  2025-01-01  Monitor        18           837.48  Veraguas  15074.64   4   
1  2025-01-02  Teclado        12           176.54    Panama   2118.48   1   
2  2025-01-03   Laptop         2           545.14  Veraguas   1090.28   4   
3  2025-01-04  Monitor        10           531.41    Panama   5314.10   1   
4  2025-01-05  Monitor         4          1407.79  Veraguas   5631.16   4   

            nombre  empleados  
0  Tienda Santiago          7  
1   Tienda Albrook         15  
2  Tienda Santiago          7  
3   Tienda Albrook         15  
4  Tienda Santiago          7  


El DataFrame unificado se guarda en formato Parquet con `to_parquet()`. Se verifica la integridad releyendo el archivo, confirmando que conserva el mismo número de filas y columnas que el DataFrame original.

In [12]:
size_csv     = os.path.getsize('ventas_simuladas.csv')
size_parquet = os.path.getsize('datos_unificados.parquet')

print(f"Tamaño CSV:     {size_csv:,} bytes")
print(f"Tamaño Parquet: {size_parquet:,} bytes")
print(f"Parquet es {size_csv/size_parquet:.1f}x más compacto que CSV")

Tamaño CSV:     4,693 bytes
Tamaño Parquet: 7,853 bytes
Parquet es 0.6x más compacto que CSV


La comparación de tamaños muestra la ventaja de compresión de Parquet frente a CSV. Aunque en datasets pequeños (100 filas) la diferencia puede ser modesta, en datasets de millones de filas el ahorro de espacio puede superar el 80%.

## Preguntas de Reflexión

### 1. ¿Qué ventajas tiene Parquet sobre CSV para grandes volúmenes?

Parquet es un formato de almacenamiento **columnar** con compresión integrada, lo que le da ventajas claras sobre CSV:

- **Compresión eficiente**: Agrupa valores del mismo tipo por columna, lo que permite comprimir mucho mejor que un CSV de texto plano. El ahorro de espacio puede ser de 5x a 10x.
- **Lectura selectiva por columna**: Si una consulta solo necesita 3 de 50 columnas, Parquet lee únicamente esas 3, ignorando el resto del archivo. CSV siempre lee fila completa.
- **Esquema embebido**: Los tipos de datos están guardados dentro del archivo, por lo que no hay riesgo de que una columna numérica sea interpretada como texto al releer.
- **Compatible con Big Data**: Es el formato nativo de Apache Spark, Hive, Google BigQuery y AWS Athena, lo que facilita procesar terabytes de datos distribuidos.

---

### 2. ¿Qué desafíos encontró al trabajar con datos de la API?

1. **Datos anidados**: JSONPlaceholder retorna campos como `address` y `company` como objetos anidados dentro del JSON. Fue necesario usar `pd.json_normalize()` para aplanar la estructura y acceder a campos como `address.city`.
2. **Formato inconsistente en REST Countries**: El campo `capital` llega como lista (`["Panama City"]`) y `name` como objeto `{"common": "...", "official": "..."}`, lo que requiere tratamiento adicional antes de poder hacer joins con otras fuentes.
3. **Dependencia de conectividad**: Las APIs públicas pueden no estar disponibles en cualquier momento, por lo que el código debe manejar errores con `try/except` y contar con datos de respaldo.
4. **Selección de campos relevantes**: Las APIs pueden retornar docenas de columnas innecesarias; fue importante filtrar solo las columnas útiles para el análisis.

---

### 3. ¿Cómo manejaría una situación donde la API no responde?

Se implementa una estrategia de **resiliencia en capas**:

```python
import time

def fetch_api_con_reintentos(url, reintentos=3, timeout=10, backoff=2):
    for intento in range(1, reintentos + 1):
        try:
            resp = requests.get(url, timeout=timeout)
            resp.raise_for_status()
            return resp.json()
        except requests.exceptions.Timeout:
            print(f"Intento {intento}: Timeout. Reintentando en {backoff**intento}s...")
            time.sleep(backoff ** intento)
        except requests.exceptions.ConnectionError:
            print(f"Intento {intento}: Sin conexión.")
            time.sleep(backoff ** intento)
        except requests.exceptions.HTTPError as e:
            print(f"Error HTTP {e}. No se reintenta.")
            break
    print("Usando caché local como respaldo.")
    return None

# Uso con fallback a caché:
data = fetch_api_con_reintentos('https://jsonplaceholder.typicode.com/users')
if data is None:
    df_api = pd.read_csv('cache_usuarios.csv')   # datos locales de respaldo
else:
    df_api = pd.json_normalize(data)
    df_api.to_csv('cache_usuarios.csv', index=False)  # actualizar caché
```

Las estrategias clave son: **reintentos con backoff exponencial** para no saturar el servidor, **caché local** con la última respuesta exitosa, y **logging** de los fallos para monitoreo.